In [1]:
from tests.test_new_ivim import _load_synthetic_test_data, _normalize_to_b0, _fit_ivim_new, _fit_ivim_reference
%matplotlib widget

In [2]:
data, mask, bvals = _load_synthetic_test_data()
signals = _normalize_to_b0(data[mask], bvals)

new_result = _fit_ivim_new(bvals, signals)
reference_result = _fit_ivim_reference(bvals, signals)

In [3]:
bvals

array([   0.,   10.,   20.,   50.,  100.,  200.,  400.,  600.,  800.,
       1000.])

In [6]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np



def to_volume(values):
    """Scatter the per-voxel results back into the image volume."""
    vol = np.full(mask.shape, np.nan)
    vol[mask] = values
    return vol


rows = [
    ("D (mm²/s)", to_volume(new_result.D), to_volume(reference_result.D)),
    ("f", to_volume(new_result.f), to_volume(reference_result.f)),
    ("D* (mm²/s)", to_volume(new_result.Dp), to_volume(reference_result.Dp)),
]

# Fixed color scales over the whole volume, so slices can be compared while scrolling.
scales = []
for _, new_vol, ref_vol in rows:
    diff_lim = max(np.nanmax(np.abs(new_vol - ref_vol)), 1e-12)
    scales.append((np.nanmin(ref_vol), np.nanmax(ref_vol), diff_lim))

slices_with_mask = np.flatnonzero(mask.any(axis=(0, 1)))


def show_slice(z):
    fig, axes = plt.subplots(len(rows), 3, figsize=(12, 10))
    for i, ((name, new_vol, ref_vol), (vmin, vmax, diff_lim)) in enumerate(zip(rows, scales)):
        new_map, ref_map = new_vol[..., z], ref_vol[..., z]

        for j, (m, title) in enumerate([(new_map, "new"), (ref_map, "reference")]):
            im = axes[i, j].imshow(m.T, vmin=vmin, vmax=vmax, cmap="viridis")
            axes[i, j].set_title(f"{name} - {title}")
            fig.colorbar(im, ax=axes[i, j], fraction=0.046)

        im = axes[i, 2].imshow((new_map - ref_map).T, vmin=-diff_lim, vmax=diff_lim, cmap="coolwarm")
        axes[i, 2].set_title(f"{name} - new minus reference")
        fig.colorbar(im, ax=axes[i, 2], fraction=0.046)

    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(f"Slice {z}  ({int(mask[..., z].sum())} masked voxels)")
    plt.tight_layout()
    plt.show()


widgets.interact(
    show_slice,
    z=widgets.IntSlider(
        value=int(slices_with_mask[len(slices_with_mask) // 2]),
        min=int(slices_with_mask.min()),
        max=int(slices_with_mask.max()),
        description="slice",
        continuous_update=False,
    ),
)


interactive(children=(IntSlider(value=4, continuous_update=False, description='slice', max=7), Output()), _dom…

<function __main__.show_slice(z)>